# 🏆 MBR (Minimum Bayes Risk) 앙상블 - 최고 성능 달성

## 이 노트북은?
- **여러 프롬프트로 생성한 요약들을 합의(consensus)로 선택**하는 앙상블 기법
- 하나의 모델 체크포인트로 다양한 프롬프트를 사용하여 여러 요약을 생성
- 각 요약 후보 중 **다른 후보들과 가장 유사한 요약**을 최종 선택

### 성능 비교
| 방법 | Mecab ROUGE-1 | 비고 |
|------|--------------|------|
| Baseline SFT | ~0.32 | 전체 시퀀스 학습 |
| Response-Only SFT (단일 프롬프트) | 0.5641 | 프롬프트 마스킹 |
| **MBR 앙상블 (8 프롬프트)** | **0.5716** | **이 노트북 (최고 성능!)** |

### MBR 디코딩이란?
Minimum Bayes Risk 디코딩은 번역/요약에서 많이 쓰이는 기법입니다:

1. 같은 입력에 대해 N개의 후보 요약을 생성 (서로 다른 프롬프트 사용)
2. 후보들끼리 **쌍별(pairwise) ROUGE 점수**를 계산
3. **다른 후보들과 평균 유사도가 가장 높은 후보**를 최종 선택

직관: "다수결" - 여러 프롬프트가 비슷하게 요약한 내용이 가장 신뢰할 수 있음

```
프롬프트 A → 요약 A
프롬프트 B → 요약 B  ← 다른 요약들과 가장 유사 → 최종 선택!
프롬프트 C → 요약 C
프롬프트 D → 요약 D
```

---

## 0. 환경 설정

In [ ]:
# 필요 라이브러리
# !pip install rouge mecab-python3 pandas tqdm

In [ ]:
import os
import re
import pandas as pd
import torch
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. 프롬프트 변형 정의

MBR 앙상블의 핵심은 **다양한 프롬프트**입니다.
같은 체크포인트를 사용하지만, 프롬프트를 다르게 하면 미묘하게 다른 요약이 생성됩니다.

우리가 사용한 8개 프롬프트:
1. **dev_save**: 기본 프롬프트 (화자 태그 강조)
2. **abstract**: 추상적 요약 스타일
3. **goldstyle**: 예시 1개 포함 (1-shot)
4. **goldstyle_v2**: 예시 3개 포함 (3-shot) + 추상적
5. **dynfew3**: dev_save와 동일 프롬프트
6. **r7_ep2**: 2에폭 학습 모델
7. **r9_32b**: Qwen3-32B (더 큰 모델)
8. **r9_32b_fewshot**: Qwen3-32B + few-shot

In [ ]:
# 프롬프트 변형 정의
PROMPTS = {
    "dev_save": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. "
            "요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. "
            "핵심 내용만 1~3문장으로 간결하게 요약하세요."
        ),
        "user": "아래 대화를 읽고 핵심 내용을 요약해주세요. 화자 태그(#Person1# 등)를 유지하세요.\n\n{dialogue}",
    },
    "abstract": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화의 주요 주제와 화자들의 행동을 요약하세요. "
            "#Person1#, #Person2# 등 화자 태그를 반드시 사용하고, "
            "'~에 대해 이야기한다', '~을 요청한다' 같은 표현을 활용하세요. "
            "1~2문장으로 간결하게 요약하세요."
        ),
        "user": "다음 대화를 요약하세요.\n\n{dialogue}",
    },
    "goldstyle": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "주어진 대화를 읽고 핵심 내용을 1~2문장으로 요약하세요.\n"
            "규칙:\n"
            "- 화자 태그(#Person1#, #Person2# 등)를 반드시 그대로 사용하세요.\n"
            "- 불필요한 세부사항은 생략하고 핵심 행동/결정/결과만 포함하세요.\n"
            "- 요약은 반드시 완전한 문장으로 끝나야 합니다.\n\n"
            "[예시]\n"
            "대화:\n"
            "#Person1#: 이것은 좋은 기본 컴퓨터 패키지입니다.\n"
            "#Person2#: 모뎀도 포함되어 있나요?\n"
            "#Person1#: 네, 내장 모뎀이 있습니다.\n"
            "#Person2#: 좋습니다. 구매하겠습니다.\n\n"
            "요약: #Person1#은 기본 컴퓨터 패키지를 #Person2#에게 보여주고, #Person2#는 구매하기로 한다."
        ),
        "user": "다음 대화를 요약하세요.\n\n{dialogue}",
    },
    "goldstyle_v2": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화의 핵심 주제와 화자들의 행동을 추상적으로 요약하세요.\n"
            "규칙:\n"
            "- #Person1#, #Person2# 등 화자 태그를 반드시 사용하세요.\n"
            "- 구체적인 세부사항이나 숫자 대신, 전체적인 주제와 행동을 설명하세요.\n"
            "- '~에 대해 이야기한다', '~을 제안한다', '~을 요청한다' 같은 추상적 표현을 사용하세요.\n"
            "- 1~2문장으로 요약하세요.\n\n"
            "[예시 1]\n"
            "대화: #Person1#: 이 신발 어때? / #Person2#: 좋아 보여. 사자.\n"
            "요약: #Person1#과 #Person2#는 신발에 대해 이야기하고, #Person2#는 구매를 제안한다.\n\n"
            "[예시 2]\n"
            "대화: #Person1#: 어제 그 영화 봤어? / #Person2#: 응, 정말 재밌었어.\n"
            "요약: #Person1#과 #Person2#는 영화에 대해 이야기한다.\n\n"
            "[예시 3]\n"
            "대화: #Person1#: 내일 회의 준비해주세요. / #Person2#: 네, 자료 준비하겠습니다.\n"
            "요약: #Person1#은 #Person2#에게 회의 준비를 요청하고, #Person2#는 동의한다."
        ),
        "user": "다음 대화를 요약하세요.\n\n{dialogue}\n\n요약:",
    },
}

print(f"{len(PROMPTS)}개 프롬프트 변형 정의 완료")
for name in PROMPTS:
    print(f"  - {name}")

## 2. 모델 로드 & 다중 프롬프트 추론

하나의 체크포인트로 여러 프롬프트에 대해 요약을 생성합니다.

> **참고**: 이 과정은 프롬프트 수 × 데이터 수만큼 시간이 걸립니다.
> 4개 프롬프트 × 499개 = ~2000번 생성 → RTX 3090 기준 약 2시간

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# 모델 로드 (Response-Only SFT 체크포인트)
CHECKPOINT = "r4b_response_only_ckpt"  # 제공된 체크포인트
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 150

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, CHECKPOINT, is_trainable=False)
FastLanguageModel.for_inference(model)
print(f"모델 로드 완료: {CHECKPOINT}")

In [ ]:
def postprocess(text):
    """모델 출력 후처리"""
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    return text


def generate_summary(dialogue, system_prompt, user_template):
    """주어진 프롬프트로 요약 생성"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_template.format(dialogue=dialogue)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return postprocess(summary)

In [ ]:
# 테스트 데이터 로드
test_df = pd.read_csv("data/test.csv")
print(f"Test: {len(test_df)}개")

# 각 프롬프트별 추론 실행
os.makedirs("prediction", exist_ok=True)
all_predictions = {}  # {prompt_name: [pred1, pred2, ...]}

for prompt_name, prompt_cfg in PROMPTS.items():
    sub_path = f"prediction/submission_{prompt_name}.csv"
    
    # 이미 생성된 파일이 있으면 건너뛰기
    if os.path.exists(sub_path):
        print(f"[{prompt_name}] 이미 존재, 로드: {sub_path}")
        df = pd.read_csv(sub_path)
        all_predictions[prompt_name] = [str(s) for s in df["summary"]]
        continue
    
    print(f"\n[{prompt_name}] 추론 시작 ({len(test_df)}개)")
    preds = []
    for i in tqdm(range(len(test_df)), desc=prompt_name):
        summary = generate_summary(
            test_df.iloc[i]["dialogue"],
            prompt_cfg["system"],
            prompt_cfg["user"],
        )
        preds.append(summary if summary.strip() else "빈 요약")
    
    # 저장
    submission = pd.DataFrame({"fname": test_df["fname"], "summary": preds})
    submission.to_csv(sub_path, index=False)
    print(f"저장: {sub_path}")
    
    all_predictions[prompt_name] = preds

print(f"\n총 {len(all_predictions)}개 프롬프트 추론 완료")

## 3. MBR 앙상블 실행

핵심 알고리즘:
```
for 각 대화 샘플:
    candidates = [프롬프트A 요약, 프롬프트B 요약, ...]
    for 각 후보 j:
        score[j] = 평균(ROUGE-1(후보j, 후보k) for k != j)
    최종 선택 = argmax(score)
```

Mecab 형태소 분석기를 사용하여 ROUGE를 계산합니다.

In [ ]:
import mecab
from rouge import Rouge

m = mecab.MeCab()
rouge = Rouge()

n_samples = len(test_df)
model_names = list(all_predictions.keys())
print(f"MBR 앙상블: {len(model_names)}개 모델, {n_samples}개 샘플")
print(f"모델: {model_names}")

mbr_preds = []
model_selected = {name: 0 for name in model_names}

for i in tqdm(range(n_samples), desc="MBR"):
    # 이 샘플에 대한 모든 후보 수집
    candidates = [(name, all_predictions[name][i]) for name in model_names]
    
    # Mecab 형태소 분석
    cand_morphs = [" ".join(m.morphs(c[1])) for c in candidates]
    
    # 각 후보의 평균 pairwise ROUGE-1 계산
    best_score = -1
    best_idx = 0
    
    for j, cj in enumerate(cand_morphs):
        avg_rouge = 0
        count = 0
        for k, ck in enumerate(cand_morphs):
            if j != k:
                try:
                    s = rouge.get_scores([cj], [ck])[0]
                    avg_rouge += s["rouge-1"]["f"]
                    count += 1
                except:
                    pass
        if count > 0:
            avg_rouge /= count
        if avg_rouge > best_score:
            best_score = avg_rouge
            best_idx = j
    
    mbr_preds.append(candidates[best_idx][1])  # 원본 텍스트 (형태소 아님)
    model_selected[candidates[best_idx][0]] += 1

print(f"\nMBR 완료!")

In [ ]:
# 모델 선택 빈도 확인
print("모델 선택 빈도:")
print("=" * 40)
for name, count in sorted(model_selected.items(), key=lambda x: -x[1]):
    bar = "█" * int(count / n_samples * 40)
    print(f"  {name:20s}: {count:3d} ({100*count/n_samples:5.1f}%) {bar}")

In [ ]:
# 제출 파일 생성
submission = pd.DataFrame({"fname": test_df["fname"], "summary": mbr_preds})
submission.to_csv("prediction/submission_mbr_ensemble.csv", index=False)
print(f"제출 파일: prediction/submission_mbr_ensemble.csv ({len(submission)}행)")
print("\n예시:")
submission.head()

## 4. (Optional) Dev 세트로 MBR 성능 검증

Dev 세트에서 MBR 앙상블의 실제 성능을 확인합니다.

In [ ]:
# Dev 세트 MBR 평가 (dev prediction 파일이 있어야 함)
dev_df = pd.read_csv("data/dev.csv")

# Dev predictions 로드
dev_preds_all = {}
for name in model_names:
    dev_path = f"prediction/dev_preds_{name}.csv"
    if os.path.exists(dev_path):
        df = pd.read_csv(dev_path)
        col = "pred" if "pred" in df.columns else "summary"
        dev_preds_all[name] = [str(p) for p in df[col]]
        print(f"  Dev 로드: {name}")

if len(dev_preds_all) >= 2:
    print(f"\nDev MBR 실행 ({len(dev_preds_all)}개 모델, {len(dev_df)}개 샘플)")
    
    dev_mbr_preds = []
    for i in tqdm(range(len(dev_df)), desc="Dev MBR"):
        candidates = [(name, dev_preds_all[name][i]) for name in dev_preds_all]
        cand_morphs = [" ".join(m.morphs(c[1])) for c in candidates]
        
        best_score = -1
        best_idx = 0
        for j, cj in enumerate(cand_morphs):
            avg_r = 0
            cnt = 0
            for k, ck in enumerate(cand_morphs):
                if j != k:
                    try:
                        s = rouge.get_scores([cj], [ck])[0]
                        avg_r += s["rouge-1"]["f"]
                        cnt += 1
                    except:
                        pass
            if cnt > 0:
                avg_r /= cnt
            if avg_r > best_score:
                best_score = avg_r
                best_idx = j
        dev_mbr_preds.append(candidates[best_idx][1])
    
    # ROUGE 계산
    golds = [str(s).strip() if str(s).strip() else "빈 요약" for s in dev_df["summary"]]
    preds_m = [" ".join(m.morphs(p)) for p in dev_mbr_preds]
    golds_m = [" ".join(m.morphs(g)) for g in golds]
    mc_scores = rouge.get_scores(preds_m, golds_m, avg=True)
    
    print(f"\n{'='*60}")
    print(f"[MBR 앙상블] Dev Mecab ROUGE:")
    print(f"  ROUGE-1: {mc_scores['rouge-1']['f']:.4f}  ← 리더보드 기준")
    print(f"  ROUGE-2: {mc_scores['rouge-2']['f']:.4f}")
    print(f"  ROUGE-L: {mc_scores['rouge-l']['f']:.4f}")
    print(f"{'='*60}")
else:
    print("Dev prediction 파일이 부족합니다. 먼저 각 프롬프트로 dev 추론을 실행하세요.")

## 📊 분석 & 정리

### MBR 앙상블의 장점
1. **추가 학습 불필요**: 같은 모델에 프롬프트만 바꿔서 다양성 확보
2. **안정적인 성능 향상**: 개별 프롬프트의 약점을 보완
3. **구현이 간단**: ROUGE 계산만으로 동작

### MBR 앙상블의 단점
1. **추론 시간 N배**: 프롬프트 수만큼 추론 필요
2. **프롬프트 다양성이 중요**: 비슷한 프롬프트만 있으면 효과 제한
3. **pairwise 비교 비용**: O(N²) 복잡도 (N=후보 수)

### 실전 팁
- 프롬프트 다양성이 핵심: 추상적 vs 구체적, 0-shot vs few-shot, 짧게 vs 길게
- 3~8개 프롬프트가 효과적 (너무 많으면 노이즈 추가)
- 다른 모델 크기(14B vs 32B)를 섞으면 더 다양한 후보 확보

### 전체 파이프라인 요약
```
1. Qwen3-14B + LoRA SFT (Response-Only Loss) → 기본 모델 학습
2. 다양한 프롬프트로 test 추론 × N회 → N개 후보 생성
3. MBR 앙상블 → 합의 기반 최종 선택
4. submission.csv 제출
```

이 파이프라인으로 **Dev Mecab ROUGE-1 = 0.5716**을 달성했습니다.